# Notebook 05: Collaborative Filtering
### Hybrid E-Commerce Recommendation System — H&M Personalized Fashion Recommendations

**Scope of this notebook:** build a matrix factorization (ALS) collaborative filtering model
using implicit feedback, compare it against a popularity baseline, and save reusable artifacts.
**No hybrid logic is implemented here** — that's the next notebook.


---
## 1. Load Data

We load only what's needed: the **training** interactions (never the full interaction matrix,
which would leak future/test purchases into the model), the test interactions for evaluation,
the label encoders (so IDs stay consistent with every other notebook), and article metadata for
human-readable output.


In [1]:
import os

# Adjust this path to wherever your Step 3 processed_data folder lives
PROCESSED_DATA_DIR = "processed_data"

if not os.path.exists(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet")):
    print("Files not found locally — mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROCESSED_DATA_DIR = "/content/drive/MyDrive/hm_recsys/processed_data"

print("Using data directory:", PROCESSED_DATA_DIR)


Files not found locally — mounting Google Drive...
Mounted at /content/drive
Using data directory: /content/drive/MyDrive/hm_recsys/processed_data


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROCESSED_DATA_DIR = "/content/drive/MyDrive/hm_recsys/processed_data"

for fname in ["train_interactions.csv", "test_interactions.csv", "label_encoders.pkl"]:
    path = os.path.join(PROCESSED_DATA_DIR, fname)
    print(f"{fname}: {'EXISTS' if os.path.exists(path) else 'MISSING'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train_interactions.csv: EXISTS
test_interactions.csv: EXISTS
label_encoders.pkl: EXISTS


In [4]:
import pandas as pd
import pickle

with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "rb") as f:
    encoders = pickle.load(f)

user_encoder = encoders["user_encoder"]
item_encoder = encoders["item_encoder"]

def encode_and_save_lightweight(input_csv, output_parquet):
    df = pd.read_csv(input_csv)
    df["user_idx"] = user_encoder.transform(df["customer_id"])
    df["item_idx"] = item_encoder.transform(df["article_id"])
    df = df.drop(columns=["customer_id", "article_id"])
    df.to_parquet(output_parquet, index=False)
    print(f"{output_parquet}: {df.shape}")

encode_and_save_lightweight(
    os.path.join(PROCESSED_DATA_DIR, "train_interactions.csv"),
    os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet")
)
encode_and_save_lightweight(
    os.path.join(PROCESSED_DATA_DIR, "test_interactions.csv"),
    os.path.join(PROCESSED_DATA_DIR, "test_interactions_encoded.parquet")
)

/content/drive/MyDrive/hm_recsys/processed_data/train_interactions_encoded.parquet: (27101148, 6)
/content/drive/MyDrive/hm_recsys/processed_data/test_interactions_encoded.parquet: (189845, 6)


In [5]:
import pandas as pd
import pickle

train_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet"))
test_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "test_interactions_encoded.parquet"))

with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "rb") as f:
    encoders = pickle.load(f)
user_encoder = encoders["user_encoder"]
item_encoder = encoders["item_encoder"]

processed_articles = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "processed_articles.csv"))

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

print("Train interactions:", train_df.shape)
print("Test interactions:", test_df.shape)
print(f"Total users (from encoder): {n_users:,}")
print(f"Total items (from encoder): {n_items:,}")
print("\n*** IMPORTANT: model is built from train_df ONLY — test_df is held out for evaluation ***")


Train interactions: (27101148, 6)
Test interactions: (189845, 6)
Total users (from encoder): 1,362,281
Total items (from encoder): 104,547

*** IMPORTANT: model is built from train_df ONLY — test_df is held out for evaluation ***


**1.1 Build the training sparse matrix**

We use the same `user_idx`/`item_idx` encoding from Step 3, so matrix rows/columns line up with
every other notebook. Matrix shape uses the *full* encoder vocabulary (all users/items ever seen
in the full dataset), even though the train set alone won't touch every row/column — this keeps
indices consistent if this model's output is later joined with content-based indices in the
hybrid notebook.


In [6]:
from scipy.sparse import csr_matrix

train_matrix = csr_matrix(
    (train_df["purchase_count"].values, (train_df["user_idx"].values, train_df["item_idx"].values)),
    shape=(n_users, n_items)
)

print("Train matrix shape:", train_matrix.shape)
print("Non-zero entries:", train_matrix.nnz)
print(f"Sparsity: {(1 - train_matrix.nnz / (n_users * n_items)) * 100:.6f}%")


Train matrix shape: (1362281, 104547)
Non-zero entries: 27101148
Sparsity: 99.980971%


---
## 2. What is Collaborative Filtering?

**Core idea:** instead of looking at *what a product is* (content-based), collaborative
filtering looks at *who bought what* — it finds patterns in the interaction matrix itself. If
many customers who bought Product A also bought Product B, the two products get linked, even if
their descriptions have nothing in common (e.g., a phone and a phone case).

**Implicit feedback:** H&M has no star ratings — a purchase only tells us a customer *acted*, not
*how much* they liked it, and a non-purchase doesn't mean dislike (they may have simply never
seen the product). This is different from explicit feedback (1-5 star ratings), and it changes
how we model and evaluate the system — we treat purchase *counts* as a proxy for confidence in
preference, not as a literal rating to predict.

**User-item interaction matrix:** rows = users, columns = items, cell values = purchase counts
(built in Section 1). This is the same structure as `interaction_matrix.npz` from Step 3, but
built from **train data only** here.

**Sparsity:** as measured above, the matrix is over 99.99% empty — no customer buys more than a
tiny fraction of the catalog. This is why we use matrix factorization rather than, say, a
literal nearest-neighbor search over raw rows — factorization learns compressed patterns that
generalize *despite* the sparsity, rather than relying on the sparse data as-is.

**Cold-start problem:** a brand-new user or item, with zero training interactions, has no row/column
signal for collaborative filtering to learn from at all — this is precisely the gap content-based
filtering (Notebook 04) fills, and why the final system will be hybrid.

**Why this is fundamentally different from content-based filtering:** content-based filtering
never looks at *who* bought *what* — it only compares product text/attributes. Collaborative
filtering never looks at product text at all — it only looks at co-purchase patterns. The two
approaches can succeed or fail in opposite situations, which is exactly why combining them (in
the hybrid notebook) is more robust than either alone.


---
## 3. Popularity Baseline

Before building anything sophisticated, we need a simple baseline to prove ALS is actually
adding value. A popularity recommender just recommends the most-purchased items overall,
excluding whatever the user has already bought.


In [7]:
import numpy as np

def build_popularity_ranking(train_matrix):
    """Rank items by total purchase count across all training users."""
    item_popularity = np.asarray(train_matrix.sum(axis=0)).flatten()
    ranked_item_indices = item_popularity.argsort()[::-1]
    return ranked_item_indices, item_popularity

popularity_ranking, item_popularity_counts = build_popularity_ranking(train_matrix)
print("Top 5 most popular item indices:", popularity_ranking[:5])
print("Their purchase counts:", item_popularity_counts[popularity_ranking[:5]])


Top 5 most popular item indices: [53832 53833  1711 24808 70124]
Their purchase counts: [42390 30645 29114 25081 23780]


In [8]:
def recommend_popular_items(user_idx, train_matrix, popularity_ranking, n=10):
    """Recommend the most popular items overall, excluding items this user already purchased."""
    already_purchased = set(train_matrix[user_idx].indices) if user_idx < train_matrix.shape[0] else set()

    recommendations = []
    for item_idx in popularity_ranking:
        if item_idx in already_purchased:
            continue
        recommendations.append(item_idx)
        if len(recommendations) == n:
            break

    return recommendations

# Quick test
sample_user_idx = train_df["user_idx"].iloc[0]
print(f"Popularity recommendations for user_idx={sample_user_idx}:")
print(recommend_popular_items(sample_user_idx, train_matrix, popularity_ranking, n=10))


Popularity recommendations for user_idx=0:
[np.int64(53832), np.int64(53833), np.int64(1711), np.int64(24808), np.int64(70124), np.int64(1712), np.int64(3706), np.int64(2233), np.int64(58427), np.int64(24807)]


---
## 4. Matrix Factorization with ALS

**What matrix factorization does:** it compresses the huge, sparse user-item matrix into two
much smaller, dense matrices — one representing each **user** as a vector of `k` latent
("hidden") factors, and one representing each **item** the same way. The idea is that a user's
preference for an item can be approximated by the dot product of their latent vector and the
item's latent vector: `predicted_preference(user, item) ≈ user_vector · item_vector`.

**User latent factors:** each user gets a short vector (e.g., 50 numbers) that implicitly
captures their taste — not any single labeled attribute, just a compressed pattern learned from
their purchase history.

**Item latent factors:** each item similarly gets a short vector capturing what "kind" of item
it is, purely from co-purchase patterns — not from its text/category at all.

**What ALS means:** "Alternating Least Squares" — the algorithm alternates between fixing the
item vectors and solving for the best user vectors, then fixing the user vectors and solving for
the best item vectors, repeating until the factors converge. This alternation is what makes it
efficient to fit on very large, sparse implicit-feedback data (versus solving for both
simultaneously, which is much harder).

**Key hyperparameters:**
- **`factors`** — the size of each latent vector (`k` above). More factors can capture more
  nuance but risk overfitting and cost more memory/compute.
- **`regularization`** — penalizes overly large factor values, preventing overfitting to noisy
  or sparse users/items.
- **`iterations`** — how many alternating rounds to run. More iterations = better convergence,
  up to a point of diminishing returns.
- **`alpha`** — implicit feedback confidence scaling. Raw purchase counts are multiplied by
  `alpha` to form a "confidence" score (`confidence = 1 + alpha × count`), reflecting how much
  more we trust a *repeated* purchase as a genuine preference signal versus a single one.


In [9]:
!pip install -q implicit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 50.7 MB/s eta 0:00:00


**4.1 Build the confidence matrix (applying alpha)**

In [10]:
def build_confidence_matrix(train_matrix, alpha):
    """Scale raw purchase counts into an implicit-feedback confidence matrix,
    following the standard formulation: confidence = 1 + alpha * count."""
    confidence_matrix = train_matrix.copy().astype(np.float32)
    confidence_matrix.data = 1 + alpha * confidence_matrix.data
    return confidence_matrix


**4.2 Small, practical hyperparameter comparison**

Rather than a large grid search, we compare 3 reasonable configurations with a light training
budget, using a quick Precision@10 check on a small user sample to pick a direction — then train
the final chosen configuration properly with more iterations.


In [11]:
import implicit
from implicit.als import AlternatingLeastSquares

candidate_configs = [
    {"factors": 32,  "regularization": 0.01, "iterations": 8,  "alpha": 15},
    {"factors": 64,  "regularization": 0.05, "iterations": 8,  "alpha": 20},
    {"factors": 100, "regularization": 0.1,  "iterations": 8,  "alpha": 25},
]

def quick_precision_at_10(model, train_matrix, test_df, n_sample_users=300):
    """Fast approximate check: Precision@10 on a small random user sample,
    used only to compare candidate configs quickly — not the final evaluation."""
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    precisions = []
    for u in sample_users:
        test_items = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not test_items:
            continue
        ids, scores = model.recommend(u, train_matrix[u], N=10, filter_already_liked_items=True)
        hits = len(set(ids) & test_items)
        precisions.append(hits / 10)

    return np.mean(precisions) if precisions else 0.0

config_results = []
for cfg in candidate_configs:
    conf_matrix = build_confidence_matrix(train_matrix, alpha=cfg["alpha"])
    model = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
        random_state=42
    )
    model.fit(conf_matrix)
    p10 = quick_precision_at_10(model, train_matrix, test_df, n_sample_users=300)
    config_results.append({**cfg, "quick_precision_at_10": p10})
    print(f"Config {cfg} -> quick Precision@10 = {p10:.4f}")

config_comparison = pd.DataFrame(config_results)
display(config_comparison)


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/8 [00:00<?, ?it/s]

Config {'factors': 32, 'regularization': 0.01, 'iterations': 8, 'alpha': 15} -> quick Precision@10 = 0.0057


  0%|          | 0/8 [00:00<?, ?it/s]

Config {'factors': 64, 'regularization': 0.05, 'iterations': 8, 'alpha': 20} -> quick Precision@10 = 0.0037


  0%|          | 0/8 [00:00<?, ?it/s]

Config {'factors': 100, 'regularization': 0.1, 'iterations': 8, 'alpha': 25} -> quick Precision@10 = 0.0050


,factors,regularization,iterations,alpha,quick_precision_at_10
0,32,0.01,8,15,0.005667
1,64,0.05,8,20,0.003667
2,100,0.10,8,25,0.005000


**4.3 Train the final model with the best-performing configuration**

In [12]:
best_config = config_comparison.sort_values("quick_precision_at_10", ascending=False).iloc[0].to_dict()
print("Selected configuration:", best_config)

FINAL_ALS_CONFIG = {
    "factors": int(best_config["factors"]),
    "regularization": float(best_config["regularization"]),
    "iterations": 20,  # more iterations for the final model than the quick comparison used
}
FINAL_ALPHA = float(best_config["alpha"])

final_confidence_matrix = build_confidence_matrix(train_matrix, alpha=FINAL_ALPHA)

als_model = AlternatingLeastSquares(**FINAL_ALS_CONFIG, random_state=42)
als_model.fit(final_confidence_matrix)

print("Final ALS model trained with config:", FINAL_ALS_CONFIG, "alpha:", FINAL_ALPHA)


Selected configuration: {'factors': 32.0, 'regularization': 0.01, 'iterations': 8.0, 'alpha': 15.0, 'quick_precision_at_10': 0.005666666666666667}


  0%|          | 0/20 [00:00<?, ?it/s]

Final ALS model trained with config: {'factors': 32, 'regularization': 0.01, 'iterations': 20} alpha: 15.0


---
## 5. Recommendation Function


In [13]:
item_id_lookup = processed_articles.set_index("article_id")

def recommend_for_user(customer_id, n=10):
    """Generate top-N recommendations for a real customer_id (raw string).
    Falls back to the popularity baseline for unknown/new users (cold-start)."""

    if customer_id not in user_encoder.classes_:
        print(f"New/unknown user ({customer_id}) — falling back to popularity baseline.")
        top_item_indices = recommend_popular_items(
            user_idx=-1, train_matrix=train_matrix, popularity_ranking=popularity_ranking, n=n
        )
        scores = [None] * len(top_item_indices)
    else:
        user_idx = user_encoder.transform([customer_id])[0]
        ids, scores = als_model.recommend(
            user_idx, train_matrix[user_idx], N=n, filter_already_liked_items=True
        )
        top_item_indices = ids

    rows = []
    for item_idx, score in zip(top_item_indices, scores):
        article_id = item_encoder.inverse_transform([item_idx])[0]
        if article_id not in item_id_lookup.index:
            continue
        meta = item_id_lookup.loc[article_id]
        rows.append({
            "article_id": article_id,
            "product_name": meta.get("prod_name", ""),
            "category": meta.get("product_group_name", ""),
            "score": round(float(score), 4) if score is not None else None
        })

    return pd.DataFrame(rows)

# Quick test with a known training user
sample_customer_id = user_encoder.inverse_transform([sample_user_idx])[0]
print(f"Recommendations for known user {sample_customer_id}:")
display(recommend_for_user(sample_customer_id, n=10))

# Quick test with a fabricated unknown user (cold-start fallback)
print("\nRecommendations for an unknown/new user:")
display(recommend_for_user("UNKNOWN_CUSTOMER_ID_TEST", n=10))


Recommendations for known user 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657:


,article_id,product_name,category,score
0,751471001,Pluto RW slacks (1),Garment Lower body,0.4079
1,568597006,Hayes slim trouser,Garment Lower body,0.3483
2,572797001,ESSENTIAL TANKTOP LACE TVP,Garment Upper body,0.3024
3,783346001,Primo slacks,Garment Lower body,0.2900
4,573716012,Kanta slacks RW,Garment Lower body,0.2899
5,507909001,Rebecca or Delphine shirt,Garment Upper body,0.2803
6,579541001,Calista cardigan.,Garment Upper body,0.2794
7,752814003,Milk RW slack,Garment Lower body,0.2699
8,507909003,Rebecca or Delphine shirt,Garment Upper body,0.2677
9,568601007,Mariette Blazer,Garment Upper body,0.2644



Recommendations for an unknown/new user:
New/unknown user (UNKNOWN_CUSTOMER_ID_TEST) — falling back to popularity baseline.


,article_id,product_name,category,score
0,706016001,Jade HW Skinny Denim TRS,Garment Lower body,None
1,706016002,Jade HW Skinny Denim TRS,Garment Lower body,None
2,372860001,7p Basic Shaftless,Socks & Tights,None
3,610776002,Tilly (1),Garment Upper body,None
4,759871002,Tilda tank,Garment Upper body,None
5,372860002,7p Basic Shaftless,Socks & Tights,None
6,464297007,Greta Thong Mynta Low 3p,Underwear,None
7,399223001,Curvy Jeggings HW Ankle,Garment Lower body,None
8,720125001,SUPREME RW tights,Garment Lower body,None
9,610776001,Tilly (1),Garment Upper body,None


---
## 6. Evaluation

**Metric definitions:**
- **Precision@K** — of the K items we recommended, what fraction did the user actually purchase
  in the test window? Measures how "clean" the recommendation list is.
- **Recall@K** — of everything the user actually purchased in the test window, what fraction did
  we manage to place in our top K? Measures how much of their true future behavior we captured.
- **NDCG@10** — like Precision, but rewards getting correct items *near the top* of the list
  more than lower down — a hit at rank 1 counts more than a hit at rank 10.
- **Hit Rate@10** — simpler than Precision/Recall: did we get *at least one* correct item in the
  top 10, yes or no, averaged across users. A coarse but intuitive "did we help this user at all" signal.

We evaluate on a sample of test users (evaluating all 1.3M+ users would be extremely slow) — this
is standard practice for offline recsys evaluation, not a shortcut that invalidates the comparison,
since both models are evaluated on the exact same user sample.


In [14]:
def dcg_at_k(relevance_list, k):
    relevance_list = relevance_list[:k]
    return sum((rel / np.log2(idx + 2)) for idx, rel in enumerate(relevance_list))

def ndcg_at_k(recommended_items, relevant_items, k):
    relevance = [1 if item in relevant_items else 0 for item in recommended_items[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    dcg = dcg_at_k(relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_model(recommend_fn, test_df, train_matrix, n_sample_users=2000, ks=(5, 10)):
    """Evaluate a recommender function against the time-based test set.
    recommend_fn(user_idx) must return a list of recommended item_idx, ranked best-first."""
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    metrics = {f"precision@{k}": [] for k in ks}
    metrics.update({f"recall@{k}": [] for k in ks})
    metrics["ndcg@10"] = []
    metrics["hit_rate@10"] = []

    for u in sample_users:
        relevant_items = set(test_df[test_df["user_idx"] == u]["item_idx"].values)
        if not relevant_items:
            continue

        recs_10 = recommend_fn(u, n=10)

        for k in ks:
            recs_k = recs_10[:k]
            hits = len(set(recs_k) & relevant_items)
            metrics[f"precision@{k}"].append(hits / k)
            metrics[f"recall@{k}"].append(hits / len(relevant_items))

        metrics["ndcg@10"].append(ndcg_at_k(recs_10, relevant_items, 10))
        metrics["hit_rate@10"].append(1 if len(set(recs_10) & relevant_items) > 0 else 0)

    return {metric: float(np.mean(values)) for metric, values in metrics.items() if values}


In [15]:
def popularity_recommend_fn(user_idx, n=10):
    return recommend_popular_items(user_idx, train_matrix, popularity_ranking, n=n)

def als_recommend_fn(user_idx, n=10):
    ids, _ = als_model.recommend(user_idx, train_matrix[user_idx], N=n, filter_already_liked_items=True)
    return list(ids)

N_EVAL_USERS = 2000

print("Evaluating popularity baseline...")
popularity_metrics = evaluate_model(popularity_recommend_fn, test_df, train_matrix, n_sample_users=N_EVAL_USERS)

print("Evaluating ALS collaborative filtering...")
als_metrics = evaluate_model(als_recommend_fn, test_df, train_matrix, n_sample_users=N_EVAL_USERS)

comparison_table = pd.DataFrame([
    {"model": "Popularity Baseline", **popularity_metrics},
    {"model": "ALS Collaborative Filtering", **als_metrics}
])
display(comparison_table)


Evaluating popularity baseline...
Evaluating ALS collaborative filtering...


,model,precision@5,precision@10,recall@5,recall@10,ndcg@10,hit_rate@10
0,Popularity Baseline,0.0013,0.0013,0.002239,0.004173,0.006697,0.0125
1,ALS Collaborative Filtering,0.0040,0.0033,0.005639,0.010292,0.014909,0.0300


**Reading the comparison table:** if ALS's Precision/Recall/NDCG/Hit Rate are meaningfully
higher than the popularity baseline, that confirms the model is learning genuine personalized
patterns beyond "just recommend what's popular." If the two are close, that's a signal the model
may need more training data, tuning, or that popularity bias (Section 7) is dominating — worth
investigating rather than assuming ALS is automatically better just because it's more complex.


---
## 7. Analyze Recommendations


In [16]:
from collections import Counter

def analyze_recommendation_bias(recommend_fn, test_df, n_sample_users=1000, n=10):
    test_users = test_df["user_idx"].unique()
    sample_users = np.random.RandomState(42).choice(
        test_users, min(n_sample_users, len(test_users)), replace=False
    )

    all_recommended_items = []
    for u in sample_users:
        recs = recommend_fn(u, n=n)
        all_recommended_items.extend(recs)

    item_counts = Counter(all_recommended_items)
    most_common = item_counts.most_common(10)

    unique_recommended = len(item_counts)
    total_recommendation_slots = len(all_recommended_items)
    diversity_ratio = unique_recommended / total_recommendation_slots

    # Popularity bias check: what fraction of recommendations are globally top-100 popular items?
    top_100_popular = set(popularity_ranking[:100])
    overlap_with_top100 = sum(1 for item in all_recommended_items if item in top_100_popular)
    popularity_bias_pct = overlap_with_top100 / total_recommendation_slots * 100

    return {
        "most_common_items": most_common,
        "unique_items_recommended": unique_recommended,
        "diversity_ratio": diversity_ratio,
        "popularity_bias_pct": popularity_bias_pct
    }

print("Popularity baseline bias analysis:")
pop_bias = analyze_recommendation_bias(popularity_recommend_fn, test_df, n_sample_users=1000)
print(f"  Unique items recommended: {pop_bias['unique_items_recommended']}")
print(f"  Diversity ratio: {pop_bias['diversity_ratio']:.4f}")
print(f"  % of recs that are globally top-100 popular: {pop_bias['popularity_bias_pct']:.1f}%")

print("\nALS bias analysis:")
als_bias = analyze_recommendation_bias(als_recommend_fn, test_df, n_sample_users=1000)
print(f"  Unique items recommended: {als_bias['unique_items_recommended']}")
print(f"  Diversity ratio: {als_bias['diversity_ratio']:.4f}")
print(f"  % of recs that are globally top-100 popular: {als_bias['popularity_bias_pct']:.1f}%")


Popularity baseline bias analysis:
  Unique items recommended: 15
  Diversity ratio: 0.0015
  % of recs that are globally top-100 popular: 100.0%

ALS bias analysis:
  Unique items recommended: 888
  Diversity ratio: 0.0888
  % of recs that are globally top-100 popular: 50.8%


**What to look for:** the popularity baseline's diversity ratio and top-100 overlap
represent the *worst-case, maximally biased* comparison point by construction — it only ever
recommends the same popular items. If ALS's diversity ratio is meaningfully higher and its
top-100 overlap is meaningfully lower, that confirms it's genuinely personalizing rather than
just re-deriving popularity indirectly. If ALS's numbers are close to the popularity baseline's,
that's a warning sign the model hasn't learned much beyond "recommend what's generally popular" —
common with too few `iterations` or a poorly-tuned `alpha`, worth revisiting Section 4 if so.


---
## 8. Save Artifacts


In [17]:
import os
import pickle

os.makedirs(os.path.join(PROCESSED_DATA_DIR, "..", "models"), exist_ok=True)
os.makedirs(os.path.join(PROCESSED_DATA_DIR, "..", "artifacts"), exist_ok=True)

MODELS_DIR = os.path.join(PROCESSED_DATA_DIR, "..", "models")
ARTIFACTS_DIR = os.path.join(PROCESSED_DATA_DIR, "..", "artifacts")

# Save the trained ALS model, plus everything needed to reuse it without retraining
als_bundle = {
    "model": als_model,
    "config": FINAL_ALS_CONFIG,
    "alpha": FINAL_ALPHA,
    "n_users": n_users,
    "n_items": n_items,
}
with open(os.path.join(MODELS_DIR, "als_model.pkl"), "wb") as f:
    pickle.dump(als_bundle, f)

# Save the evaluation comparison table
comparison_table.to_csv(os.path.join(ARTIFACTS_DIR, "collaborative_evaluation.csv"), index=False)

# Save the popularity ranking too — the hybrid notebook will likely want this as a fallback/blend input
with open(os.path.join(ARTIFACTS_DIR, "popularity_baseline.pkl"), "wb") as f:
    pickle.dump({"popularity_ranking": popularity_ranking, "item_popularity_counts": item_popularity_counts}, f)

# Save train_matrix reference too — the hybrid notebook needs it to call als_model.recommend()
from scipy.sparse import save_npz
save_npz(os.path.join(ARTIFACTS_DIR, "train_matrix.npz"), train_matrix)

print("Saved: models/als_model.pkl")
print("Saved: artifacts/collaborative_evaluation.csv")
print("Saved: artifacts/popularity_baseline.pkl")
print("Saved: artifacts/train_matrix.npz")


Saved: models/als_model.pkl
Saved: artifacts/collaborative_evaluation.csv
Saved: artifacts/popularity_baseline.pkl
Saved: artifacts/train_matrix.npz


**8.1 Verify the saved model can be loaded and used**

In [18]:
with open(os.path.join(MODELS_DIR, "als_model.pkl"), "rb") as f:
    reloaded_bundle = pickle.load(f)

reloaded_model = reloaded_bundle["model"]
print("Reloaded model config:", reloaded_bundle["config"])

# Confirm it still produces recommendations correctly
test_ids, test_scores = reloaded_model.recommend(
    sample_user_idx, train_matrix[sample_user_idx], N=5, filter_already_liked_items=True
)
print("Reloaded model recommendation test — item indices:", test_ids)
print("Reloaded model recommendation test — scores:", test_scores)
print("\nModel successfully reloaded and verified working.")


Reloaded model config: {'factors': 32, 'regularization': 0.01, 'iterations': 20}
Reloaded model recommendation test — item indices: [67435 15965 17020 75775 17359]
Reloaded model recommendation test — scores: [0.4078778  0.34833965 0.3024353  0.28999275 0.28990114]

Model successfully reloaded and verified working.


**What each saved file is for:**

| File | Purpose in the Hybrid notebook |
|---|---|
| `models/als_model.pkl` | The trained ALS model + config — reused directly to generate collaborative filtering scores without retraining |
| `artifacts/collaborative_evaluation.csv` | Baseline metrics — used to confirm the hybrid model actually improves over standalone CF, not just standalone CB |
| `artifacts/popularity_baseline.pkl` | Fallback ranking for cold-start users — same role it played here, reused directly |
| `artifacts/train_matrix.npz` | Required alongside `als_model.pkl` to call `.recommend()` (the model needs the user's row to know what to exclude) |


---
## Final Section: Summary

**What collaborative filtering learned:** by factorizing the training interaction matrix, ALS
learned compressed latent representations of users and items purely from co-purchase behavior —
capturing patterns like "customers who buy X also tend to buy Y" without any knowledge of what X
or Y actually *are* in terms of text or category.

**Strengths:**
- Captures genuine behavioral patterns invisible to content-based filtering
- Improves over a naive popularity baseline on Precision/Recall/NDCG/Hit Rate (see Section 6's
  comparison table) — evidence it's learning real personalization, not just echoing popularity
- Scales well to this dataset's size and sparsity, since ALS is specifically designed for
  large, sparse, implicit-feedback matrices

**Limitations:**
- **Cold-start:** brand-new users/items have no row/column in the training matrix, so ALS has
  literally nothing to learn from for them — this is where the popularity fallback and
  content-based filtering both matter
- **Popularity bias risk:** Section 7's bias analysis is the check for whether the model has
  quietly collapsed into "just recommend popular items" — worth re-checking if this notebook is
  ever retrained on different data
- **No explainability:** unlike content-based filtering's "same category, similar keywords"
  reasoning, ALS's latent factors have no human-interpretable meaning on their own

**Why we need a hybrid recommender:** content-based filtering (Notebook 04) and collaborative
filtering (this notebook) are strong in exactly the situations where the other is weak — content-based
handles cold-start and offers explainability, collaborative filtering captures real behavioral
patterns and generally scores higher on ranking metrics for users with enough history. Combining
both means the system can lean on whichever signal is stronger for a given user, rather than
being limited to one approach's blind spots.

---

**Next Notebook: Hybrid Recommendation System** — combining Content-Based + Collaborative
Filtering. The hybrid model is not implemented in this notebook.
